In [2]:
%mamba install requests

%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, requests, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.110 seconds
All requested packages already installed.
mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, requests, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.121 seconds
All requested packages already installed.


In [16]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

In [17]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji", "Park Jisoo", "Lee Jaehyun", "Choi Seoyeon", "Jung Hyunwoo",
         "Kang Yuna", "Yoon Doyoon", "Han Soomin", "Oh Jiwoo", "Shin Taeyang",
         "Baek Narae", "Song Junho", "Moon Hana", "Ahn Jiho", "Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


In [19]:
def customers_in_city_segment(conn, city, segment):
    """
    conn: sqlite3 connection
    city: str, e.g. "Seoul"
    segment: str, e.g. "Online"
    returns: pandas DataFrame with columns name, city, segment
    """
    query = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql(query, conn, params=(city, segment))

# Test
print(customers_in_city_segment(conn, "Busan", "Retail"))

           name   city segment
0   Yoon Doyoon  Busan  Retail
1  Shin Taeyang  Busan  Retail


In [20]:
def orders_with_customer_names(conn):
    """
    returns: pandas DataFrame with columns name, order_date, amount (first 20 rows)
    """
    query = """
        SELECT c.name, o.order_date, o.amount
        FROM orders o
        INNER JOIN customers c ON o.customer_id = c.customer_id
        LIMIT 20
    """
    return pd.read_sql(query, conn)

print(orders_with_customer_names(conn))

            name  order_date    amount
0      Kim Minji  2026-01-26  439700.0
1      Kim Minji  2026-01-24  257100.0
2      Kim Minji  2026-03-10  141000.0
3      Kim Minji  2026-03-15  250200.0
4      Kim Minji  2026-06-07  258700.0
5     Park Jisoo  2026-04-03   59800.0
6     Park Jisoo  2026-04-07  177000.0
7     Park Jisoo  2026-02-02  260500.0
8     Park Jisoo  2026-04-16  230900.0
9     Park Jisoo  2026-08-07  353100.0
10    Park Jisoo  2026-08-27  269700.0
11    Park Jisoo  2026-08-21  172300.0
12   Lee Jaehyun  2026-04-03  319100.0
13   Lee Jaehyun  2026-05-05   50600.0
14   Lee Jaehyun  2026-06-03  243500.0
15  Choi Seoyeon  2026-08-18  140300.0
16  Choi Seoyeon  2026-02-07   66400.0
17  Choi Seoyeon  2026-08-03   86800.0
18  Choi Seoyeon  2026-06-25   81100.0
19  Jung Hyunwoo  2026-08-04   32100.0


In [21]:
def high_spending_customers(conn, min_total=250000):
    """
    returns: pandas DataFrame with columns name, total_spent
    """
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        HAVING SUM(o.amount) > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql(query, conn, params=(min_total,))

print(high_spending_customers(conn))

            name  total_spent
0     Jang Yerin    2167100.0
1    Yoon Doyoon    1609600.0
2     Baek Narae    1559500.0
3     Park Jisoo    1523300.0
4       Oh Jiwoo    1437300.0
5     Song Junho    1359000.0
6      Kim Minji    1346700.0
7       Ahn Jiho    1129700.0
8      Moon Hana     895700.0
9   Jung Hyunwoo     861200.0
10  Shin Taeyang     619200.0
11   Lee Jaehyun     613200.0
12    Han Soomin     427900.0
13  Choi Seoyeon     374600.0
14     Kang Yuna     262600.0


In [24]:
def top_n_customers(conn, n=5):
    """
    returns: pandas DataFrame with columns name, total_spent - exactly n rows
    """
    query = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers c
        INNER JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql(query, conn, params=(n,))

print(top_n_customers(conn, 5))

          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0
3   Park Jisoo    1523300.0
4     Oh Jiwoo    1437300.0


In [28]:
def find_orders_by_customer(conn, customer_name):
    """
    conn: sqlite3 connection
    customer_name: str
    returns: list of tuples (order_id, order_date, amount)
    """
    sql = """
        SELECT o.order_id, o.order_date, o.amount
        FROM orders o
        INNER JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.name = ?
    """
    cur = conn.cursor()
    cur.execute(sql, (customer_name,))
    return cur.fetchall()

print(find_orders_by_customer(conn, "Park Jisoo"))

[(6, '2026-04-03', 59800.0), (7, '2026-04-07', 177000.0), (8, '2026-02-02', 260500.0), (9, '2026-04-16', 230900.0), (10, '2026-08-07', 353100.0), (11, '2026-08-27', 269700.0), (12, '2026-08-21', 172300.0)]


In [29]:
def monthly_revenue(conn):
    """
    returns: pandas DataFrame with columns month, revenue
    """
    query = """
        SELECT strftime('%Y-%m', order_date) AS month,
               SUM(amount) AS revenue
        FROM orders
        GROUP BY month
        ORDER BY month
    """
    return pd.read_sql(query, conn)

print(monthly_revenue(conn))

     month    revenue
0  2026-01  1605200.0
1  2026-02  1924900.0
2  2026-03  1836600.0
3  2026-04  2277600.0
4  2026-05  1005400.0
5  2026-06  2817100.0
6  2026-07  1715100.0
7  2026-08  1906900.0
8  2026-09  1097800.0


In [30]:
def get_current_weather(latitude, longitude):
    """
    returns: dict (the current_weather portion of the response), or None on failure
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": "true",
    }
    try:
        response = requests.get(url, params=params, timeout=10)
    except requests.exceptions.RequestException:
        return None

    if response.status_code != 200:
        return None

    data = response.json()
    return data.get("current_weather")

print(get_current_weather(37.5665, 126.9780))  # Seoul

{'time': '2026-10-11T02:00', 'interval': 900, 'temperature': 21.3, 'windspeed': 2.2, 'winddirection': 81, 'is_day': 1, 'weathercode': 0}


In [31]:
def weather_for_cities(cities):
    """
    cities: list of (city_name, latitude, longitude) tuples
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    rows = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        if weather is None:
            continue  # skip cities whose request failed
        row = {"city": city_name}
        row.update(weather)
        rows.append(row)
        time.sleep(0.2)  # be polite to the free API
    return pd.DataFrame(rows)

test_cities = [("Seoul", 37.5665, 126.9780), ("Busan", 35.1796, 129.0756), ("Tokyo", 35.6762, 139.6503)]
print(weather_for_cities(test_cities))

    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-11T02:00       900         21.3        2.2             81   
1  Busan  2026-10-11T02:00       900         24.9        9.0             90   
2  Tokyo  2026-10-11T02:00       900         23.5        5.4             20   

   is_day  weathercode  
0       1            0  
1       1            1  
2       1            0  


In [33]:
def weather_for_cities_paced(cities, delay_seconds=1):
    """
    Same as Problem 8, but pauses delay_seconds between each request.
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    rows = []
    for city_name, lat, lon in cities:
        weather = get_current_weather(lat, lon)
        time.sleep(delay_seconds)  # pause after every request, even failed ones
        if weather is None:
            continue
        row = {"city": city_name}
        row.update(weather)
        rows.append(row)
    return pd.DataFrame(rows)

print(weather_for_cities_paced(test_cities, delay_seconds=1))

    city              time  interval  temperature  windspeed  winddirection  \
0  Seoul  2026-10-11T02:00       900         21.3        2.2             81   
1  Busan  2026-10-11T02:00       900         24.9        9.0             90   
2  Tokyo  2026-10-11T02:00       900         23.5        5.4             20   

   is_day  weathercode  
0       1            0  
1       1            1  
2       1            0  


In [15]:
conn.close()
print("Connection closed.")

Connection closed.
